In [ ]:
from google.colab import drive
# Mount Google Drive to access files stored in your Drive.
# This is typically needed in Google Colab to persist data and code.
drive.mount('/content/drive')

In [ ]:
import os

# For reproducibility on GitHub or a local environment:
# Set BASE_PATH to the current working directory. This assumes your project
# structure starts from the directory where this notebook is located.
# If your project needs a specific subdirectory as its root, you can change
# it to, e.g., `os.path.join(os.getcwd(), 'my_project_folder')`.
#
# To use Google Drive in Colab (as previously configured), you would uncomment
# the `drive.mount` cell (fArm4cgMvSFa) and set BASE_PATH to:
# BASE_PATH = "/content/drive/MyDrive/pw_analysis"
#
# For this GitHub-friendly version, we'll use the current directory.
BASE_PATH = os.getcwd()

# Ensure the base path exists before changing directory
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)
print(f"Current working directory changed to: {os.getcwd()}")

Essential libraries for geospatial data processing and data manipulation (geopandas, shapely, pyproj, fiona), installed using pip.

In [ ]:
# Install essential libraries for geospatial data processing and manipulation.
# geopandas: Extends pandas for geospatial data, handling geographic data types.
# shapely: For geometric objects like points, lines, and polygons.
# pyproj: Performs cartographic projections and coordinate transformations.
# fiona: Reads and writes geospatial data files.
!pip install geopandas shapely pyproj fiona

In [ ]:
import pandas as pd
import geopandas as gpd

# Print the versions of Pandas and GeoPandas to ensure compatibility and for debugging purposes.
print("Pandas version:", pd.__version__)
print("GeoPandas version:", gpd.__version__)

In [ ]:
# Imports necessary libraries for fetching and visualizing OCD (Oil Conservation Division) well data from ArcGIS.
# requests: For making HTTP requests to the ArcGIS FeatureServer.
# shapely.geometry.Point: For creating point objects from coordinates.
# folium: For creating interactive maps.
import requests
from shapely.geometry import Point
import folium

In [ ]:
# Imports necessary libraries for processing and visualizing shapefiles.
# folium: For creating interactive maps.
# folium.plugins.MarkerCluster: For clustering markers on the map when there are many points.
# os: For interacting with the operating system, like managing file paths.
import folium
from folium.plugins import MarkerCluster
import os

# **WaterSTAR wells**
Data quality and quantity = for all 267 quarter townships

Data fetched from a web API (https://nmpw.waterstar.org/api/api/well) by making POST requests. The process involved looping through multiple 'well pages' (from 1 to 6), collecting all data, and then converting it into a pandas DataFrame.

In [ ]:
import requests
import pandas as pd

all_data = []

# Base URL for the API call to fetch well data from WaterSTAR.
base_api_url = "https://nmpw.waterstar.org/api/api/well"

# Headers extracted from a cURL command. These are necessary to mimic a browser request
# and typically include user-agent, referer, cookies, and other authentication/session info.
headers = {
    'accept': '*/*',
    'accept-language': 'en-US,en;q=0.9',
    'access-control-allow-headers': 'Access-Control-Allow-Origin, X-Requested-With, Content-Type, Accept',
    'access-control-allow-origin': '*',
    'authorization': 'Bearer null', # May indicate no specific authentication token is required or is handled differently.
    'content-type': 'application/json',
    'dnt': '1',
    'origin': 'https://nmpw.waterstar.org',
    'priority': 'u=1, i',
    'referer': 'https://nmpw.waterstar.org/?wellPageSize=50&wellPage=1&samplePageSize=50&samplePage=1&',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    # The 'Cookie' header contains session-specific information and tracking IDs.
    'Cookie': '_ga=GA1.1.1480337763.1774023695; AMP_MKTG_8f1ede8e9c=JTdCJTIycmVmZXJyZXIlMjIlM0ElMjJodHRwcyUzQSUyRiUyRm5tcHdyYy5ubXN1LmVkdCUyRiUyMiUyQyUyMnJlZmVycmluZ19kb21haW4lMjIlM0ElMjJubXB3cmMubm1zdS5lZHUlN0Q=; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJlMmUzNzYzZS00YTMzLTRjZTItOThkMi0wM2QwOTE5ZDQ5ZWElMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0MjU5NDQxMzU3JTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlMkMlMjJsYXN0RXZlbnRUaW1lJTIyJTNBMTc0MjU5NDQxNDYxOCU3RA==; _ga_3DB96KXQHX=GS2.1.s1774449888$o10$g1$t1774452691$j59$l0$h0; _ga_S0M0XQD1W9=GS2.1.s1774515451$o7$g1$t1774515860$j59$l0$h0; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJlMmUzNzYzZS00YTMzLTRjZTItOThkMi0wM2QwOTE5ZDQ5ZWFlJTIyJTJDJTIyc2Vzc2lvbklkJTIyJTNBMTc3NDUyMTgwOTQyNyUyQyUyMm9wdE91dCUyMiUzQWZhbHNlJTJDJTIybGFzdEV2ZW50VGltZSUyMiUzQTE3NzQ1MjE4MDk0MjclN0Q==; _ga_9H1TP8FD9K=GS2.1.s1774521619$o14$g1$t1774521848$j19$l0$h0'
}

# The request body (payload) sent with the POST request.
# In this case, it's an empty object, indicating no specific filters or data are being sent in the body.
request_json_payload = {"selectedIds":""}

# Loop through well pages 1 to 6 to retrieve data in batches.
# Each page fetches a maximum of 50 wells, so this will get up to 300 wells.
for page_num in range(1, 7): # Looping through well pages 1 to 6
    # Query parameters to control pagination and sample size for the API request.
    params = {
        "wellPageSize": 50,  # Number of wells to return per page.
        "wellPage": page_num, # Dynamically change wellPage for each iteration.
        "samplePageSize": 50,
        "samplePage": 1,      # samplePage seems to be fixed at 1 based on the curl command structure.
    }

    try:
        # Make a POST request to the API.
        # - base_api_url: The endpoint to send the request to.
        # - headers: HTTP headers for the request.
        # - json: The JSON payload for the request body.
        # - params: URL query parameters.
        response = requests.post(base_api_url, headers=headers, json=request_json_payload, params=params)
        # Raise an HTTPError for bad responses (4xx or 5xx status codes).
        response.raise_for_status()
        # Parse the JSON response into a Python dictionary.
        data = response.json()

        # Check if the 'data' key exists in the response and contains a list of records.
        if "data" in data and isinstance(data[

In [ ]:
df.info()

# **Automating Data Scraping for All Quarter Townships**

for all 267 quarter townships, all the `detailKey` values = (e.g., `15112` for '024S 026E NE').

Reads a list of QTSID (Quarter Township IDs) from a wells_data.csv file. It then loops through each ID, making an API request to retrieve quantity data for all 267 townships.

In [ ]:
import requests
import pandas as pd

# --- Placeholder for the list of all detailKeys (Quarter Township IDs) ---
# This list (qtsid_list) needs to be defined from a previous step or loaded from a file.
# It's crucial for iterating through all 267 townships.
all_quarter_township_detail_keys = qtsid_list


# Base URL and headers remain the same as the successful individual call
base_url = "https://nmpwdashboard.waterstar.org/api/api/widget/278"
# Headers are critical for mimicking a valid request to the API.
headers = {
    'accept': '*/*',
    'accept-language': 'en-US,en;q=0.9',
    'dnt': '1',
    'priority': 'u=1, i',
    'referer': 'https://nmpwdashboard.waterstar.org/NM%20Produced%20Water/qt-details/15112',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'Cookie': '_ga=GA1.1.1480337763.1774023695; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJlMmUzNzYzZS00YTMzLTRjZTItOThkMi0wM2QwOTE5ZWFlJTIyJTJDJTIyc2Vzc2lvbklkJTIyJTNBMTc3NDI1OTQ0MTM1NyUyQyUyMm9wdE91dCUyMiUzQWZhbHNlJTdE; _ga_3DB96KXQHX=deleted; _ga_3DB96KXQHX=deleted; _ga_S0M0XQD1W9=GS2.1.s1774605931$o8$g0$t1774605933$j58$l0$h0; AMP_MKTG_8f1ede8e9c=JTdCJTIycmVmZXJyZXIlMjIlM0ElMjJodHRwcyUzQSUyRiUyRm5tLndhdGVyc3Rhci5vcmclMkYlMjIlMkMlMjJyZWZlcnJpbmdfZG9tYWluJTIyJTNBJTIybm0ud2F0ZXJzdGFyLm9yZyUyMiU3RA==; _ga_9H1TP8FD9K=GS2.1.s1774605810$o19$g1$t1774605977$j6$l0$h0; _ga_3DB96KXQHX=GS2.1.s1774605570$o13$g1$t1774606028$j38$l0$h0; AMP_8f1ede8e9c=JTdCJTIyZGVpY2VJZCUyMiUzQSUyMmUyZTM3NjNlLTRhMzMtNGNlMi05OGQyLTAzZDA5MTllYWUlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0MjU5NDQxMzU3JTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q'
}

all_township_data = []

# Iterate through each Quarter Township ID (detailKey).
# For each detailKey, an API request is made to fetch time-series data.
for detail_key in all_quarter_township_detail_keys:
    # Query parameters for the API call.
    params = {
        'dateTo': '03/27/2026', # Specifies the end date for the data query.
        'detailKey': detail_key # The unique identifier for the specific township.
    }

    try:
        # Make a GET request to the API with the defined headers and parameters.
        response = requests.get(base_url, headers=headers, params=params)
        # Raise an HTTPError for bad responses (4xx or 5xx status codes).
        response.raise_for_status()
        # Parse the JSON response into a Python dictionary.
        data = response.json()

        # Check if the 'data' key and 'Data' sub-key exist in the response.
        # The actual time-series data is expected to be under 'data['Data']'.
        if 'data' in data and 'Data' in data['data']:
            # Convert the list of dictionaries under 'Data' into a pandas DataFrame.
            df_temp = pd.DataFrame(data['data']['Data'])
            all_township_data.append(df_temp)
            print(f"Successfully fetched data for detailKey: {detail_key}")
        else:
            # Handle cases where the expected 'Data' key is missing in the response.
            print(f"Warning: 'Data' key not found in response for detailKey: {detail_key}")
            print(data)

    except requests.exceptions.RequestException as e:
        # Catch and report any network-related errors during the request.
        print(f"Error fetching data for detailKey {detail_key}: {e}")
    except ValueError as e:
        # Catch and report errors if the response content is not valid JSON.
        print(f"Error decoding JSON response for detailKey {detail_key}: {e}")
        print(f"Raw response content: {response.text if 'response' in locals() else 'No response object'}")

# After looping through all detailKeys, concatenate all collected DataFrames.
if all_township_data:
    # Concatenate all individual DataFrames (each representing a township's data)
    # into one large DataFrame for comprehensive analysis.
    df_all_townships = pd.concat(all_township_data, ignore_index=True)
    print(f"\nSuccessfully collected data for {len(all_township_data)} townships.")
    print(f"Final DataFrame shape: {df_all_townships.shape}")
    # Display the first few rows of the combined DataFrame.
    display(df_all_townships.head())
else:
    print("No data was collected.")

Concatenates all the collected individual township data into a single comprehensive pandas DataFrame (df_all_townships). Finally, it saves this aggregated data to a CSV file named all_quarter_townships_quantity_data.csv in your Google Drive.

In [ ]:
import os

# Define the directory path to save the cleaned data. It uses the BASE_PATH
# to ensure the directory structure is relative and reproducible.
output_dir = os.path.join(BASE_PATH, "data_clean")

# Create the output directory if it does not already exist.
# exist_ok=True prevents an error if the directory already exists.
os.makedirs(output_dir, exist_ok=True)

# Define the full path for the output CSV file.
output_file_path = os.path.join(output_dir, "all_quarter_townships_quantity_data.csv")

# Save the DataFrame (df_all_townships) to a CSV file.
# index=False prevents writing the DataFrame index as a column in the CSV.
df_all_townships.to_csv(output_file_path, index=False)

print(f"DataFrame saved to: {output_file_path}")

# NM OCD wells

**OCD data via ArcGIS**

It defines a function fetch_ocd to query the ArcGIS FeatureServer for well data. It then paginates through the entire dataset, fetching all records (over 140,000) and reporting the progress.

In [ ]:
# The URL for the ArcGIS FeatureServer endpoint containing well data.
SERVICE_URL = (
    "https://gis.emnrd.nm.gov/arcgis/rest/services/OCDView/Wells_Public/FeatureServer/0/query" # The 'worked' comment indicates this URL was previously validated.
)

# Define a function to fetch data from the ArcGIS FeatureServer.
# where: SQL-like WHERE clause for filtering features (e.g., "1=1" means all features).
# offset: The starting record for pagination.
# batch: The number of records to fetch per request.
def fetch_ocd(where="1=1", offset=0, batch=2000):
    params = {
        "where": where, # Filter clause
        "outFields": "*", # Request all fields (attributes) for each feature.
        "f": "json", # Specify JSON as the output format.
        "resultOffset": offset, # Pagination offset.
        "resultRecordCount": batch, # Max number of records to return.
        "geometryType": "esriGeometryPoint", # Specify the geometry type to return.
        "spatialRel": "esriSpatialRelIntersects", # Spatial relationship to consider (e.g., intersects).
        "outSR": "4326", # Output spatial reference (WGS 84, common for lat/lon).
        "returnGeometry": "true", # Request geometry information for each feature.
    }
    # Make a GET request to the ArcGIS service.
    r = requests.get(SERVICE_URL, params=params, timeout=30)
    # Raise an exception for HTTP errors (4xx or 5xx responses).
    r.raise_for_status()
    # Return the JSON response.
    return r.json()

# Paginate through all records to retrieve the complete dataset.
all_features = [] # List to store all fetched features.
offset = 0 # Initialize the offset for the first request.
while True:
    # Fetch a batch of data.
    data = fetch_ocd(offset=offset)
    features = data.get("features", []) # Extract features; default to empty list if not found.
    if not features:
        break # Exit loop if no features are returned, indicating end of data.
    all_features.extend(features) # Add fetched features to the master list.
    offset += len(features) # Increment offset by the number of features just fetched.
    print(f"  fetched {len(all_features)} records so far...") # Progress indicator.
    # Check if the service indicates there are more records beyond the transfer limit.
    # If not, all records have been retrieved.
    if not data.get("exceededTransferLimit", False):
        break

print(f"Total records: {len(all_features)}")

  fetched 30000 records so far...
  fetched 32000 records so far...
  fetched 34000 records so far...
  fetched 36000 records so far...
  fetched 38000 records so far...
  fetched 40000 records so far...
  fetched 42000 records so far...


The fetched JSON data is parsed into a pandas DataFrame. Missing longitude/latitude values are dropped, and the DataFrame is converted into a GeoDataFrame, which is suitable for geographical operations, using EPSG:4326 as the coordinate reference system.

In [ ]:
# Parsing the fetched ArcGIS JSON data into a pandas DataFrame and then a GeoDataFrame.
records = [] # List to hold processed well records.
for f in all_features:
    attr = f["attributes"] # Extract the attribute dictionary.
    geom = f.get("geometry", {}) # Extract geometry dictionary, default to empty if not present.
    attr["longitude"] = geom.get("x") # Extract longitude (x-coordinate) from geometry.
    attr["latitude"]  = geom.get("y") # Extract latitude (y-coordinate) from geometry.
    records.append(attr) # Add the enriched attribute dictionary to the records list.

df = pd.DataFrame(records) # Create an initial pandas DataFrame from the records.
print("Columns:", df.columns.tolist()) # Print all column names for review.
print("Well types:", df["type"].value_counts().head(10)) # Show the distribution of well types.

# Convert to GeoDataFrame — drop rows missing coordinates as they cannot be mapped.
df = df.dropna(subset=["longitude", "latitude"]) # Remove rows where longitude or latitude is missing.
gdf = gpd.GeoDataFrame(
    df, # Base DataFrame.
    geometry=gpd.points_from_xy(df.longitude, df.latitude), # Create Point geometry objects.
    crs="EPSG:4326" # Set the Coordinate Reference System to WGS 84 (latitude/longitude).
)

print(f"GeoDataFrame shape: {gdf.shape}") # Print the dimensions of the GeoDataFrame.
gdf.head(3) # Display the first 3 rows of the GeoDataFrame.

The data is filtered to separate 'Oil' wells and 'Salt Water Disposal' (SWD) wells based on their 'type' attribute. It also prints the count of each type.

In [ ]:
# 3. Split the GeoDataFrame into Oil wells and SWD (Salt Water Disposal) wells.
# This step categorizes wells based on their 'type' attribute.

# First, print unique values in the 'type' column to understand the exact strings used,
# which helps in creating accurate filters.
print(df["type"].unique())   # adjust filter below to match

# Filter for oil wells: selects rows where the 'type' column (case-insensitive) contains "OIL".
# na=False ensures that NaN values in 'type' do not raise errors and are treated as False.
oil_wells = gdf[gdf["type"].str.upper().str.contains("OIL",  na=False)]
# Filter for SWD wells: selects rows where the 'type' column (case-insensitive) contains "WATER".
# This assumes 'WATER' in the type string indicates a Salt Water Disposal well.
swd_wells = gdf[gdf["type"].str.upper().str.contains("WATER", na=False)]

# Print the counts of identified oil and SWD wells.
print(f"Oil wells: {len(oil_wells):,}")
print(f"SWD wells: {len(swd_wells):,}")

creates an interactive map using folium. SWD wells are marked with teal circles, and a sample of oil wells are marked with smaller grey dots, allowing for a visual representation of their distribution. The map is saved as an HTML file and displayed inline.

In [ ]:
# Create an interactive map using Folium to visualize the distribution of wells.
# Initialize the map centered on New Mexico (approx. 34.5 lat, -106.0 lon) with a zoom level of 7.
# 'CartoDB positron' is chosen for a light-themed base map.
m = folium.Map(location=[34.5, -106.0], zoom_start=7, tiles="CartoDB positron")

# Add SWD wells to the map as teal circles.
# Iterate through each SWD well in the 'swd_wells' GeoDataFrame.
for _, row in swd_wells.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x], # Latitude and Longitude for marker placement.
        radius=3, # Small radius for individual well markers.
        color="#1D9E75", # Teal color for SWD wells.
        fill=True, fill_opacity=0.7, # Fill the circles with some transparency.
        popup=f"SWD: {row.get('WellName','')}" # Display well name on click.
    ).add_to(m) # Add the marker to the map object.

# Add Oil wells to the map as small grey dots.
# A sample of oil wells is used (max 3000) to prevent overcrowding the map due to many thousands of wells.
for _, row in oil_wells.sample(min(3000, len(oil_wells))).iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1, # Very small radius for oil wells.
        color="#888780", # Grey color for oil wells.
        fill=True, fill_opacity=0.4 # Fill with transparency.
    ).add_to(m)

# Add a layer control to the map, allowing users to toggle different layers (if defined).
folium.LayerControl().add_to(m)
# Save the generated map as an HTML file.
m.save("ocd_wells_map.html")
# Display the map inline in the Colab notebook.
m   # renders inline in Colab

# **NM county shapefile**

Downloaded from: [Tigerline Shape file](https://catalog.data.gov/dataset/tiger-line-shapefile-2022-nation-u-s-county-and-equivalent-entities)

In [ ]:
# Define working directories relative to the BASE_PATH for reproducibility.
# BASE_PATH is set in a previous cell (e.g., to `/content` in Colab or current directory locally).
# WORK_DIR is set to BASE_PATH, assuming the project structure starts here.
WORK_DIR = BASE_PATH
# DATA_DIR is a subdirectory for raw data within the WORK_DIR.
DATA_DIR = os.path.join(WORK_DIR, "data_raw")
# SHP_DIR is a specific subdirectory for shapefiles within DATA_DIR.
SHP_DIR  = os.path.join(DATA_DIR, "tl_2022_county")

In [ ]:
# Locate the ESRI Shapefile (.shp) within the SHP_DIR.
# It assumes there is only one .shp file in the directory.
shp_file = [f for f in os.listdir(SHP_DIR) if f.endswith(".shp")][0]
# Construct the full path to the shapefile.
shp_path = os.path.join(SHP_DIR, shp_file)
print(f"Loading: {shp_file}")

In [ ]:
# Load the county shapefile into a GeoDataFrame and reproject it to EPSG:4326.
# gpd.read_file(shp_path): Reads the shapefile.
# .to_crs("EPSG:4326"): Converts the coordinate reference system to WGS 84 (latitude/longitude).
nm_counties = gpd.read_file(shp_path).to_crs("EPSG:4326")

print(f"Columns after loading: {nm_counties.columns.tolist()}")

# Filter the GeoDataFrame to include only counties for New Mexico (STATEFP == "35").
nm_counties = nm_counties[nm_counties["STATEFP"] == "35"]
# Create a new 'county_key' column for easier merging or identification, converting county names to lowercase.
nm_counties["county_key"] = nm_counties["NAME"].str.strip().str.lower()

print(f"\n✓ Loaded: {len(nm_counties)} counties")
print(f"CRS: {nm_counties.crs}")

print(nm_counties[["NAME","county_key"]].head())

In [ ]:
# Verify the number of rows (counties) and the Coordinate Reference System (CRS).
# For New Mexico, there should be 33 counties.
print(f"Rows:    {len(nm_counties)}")    # Must be 33 counties for New Mexico.
print(f"CRS:     {nm_counties.crs}")     # Must be EPSG:4326 for consistency with other spatial data.

In [ ]:
import os

# Define the path for the GeoJSON cache file within the DATA_DIR.
GEOJSON_PATH = os.path.join(DATA_DIR, "nm_counties.geojson")

# Check if the GeoJSON file already exists to avoid reprocessing.
if not os.path.exists(GEOJSON_PATH):
    # If the file does not exist, save the 'nm_counties' GeoDataFrame to GeoJSON format.
    nm_counties.to_file(GEOJSON_PATH, driver="GeoJSON")
    print(f"✓ Saved: nm_counties.geojson")
else:
    # If the file exists, load it from the cache to save time.
    print("✓ GeoJSON already exists — loading from cache")
    nm_counties = gpd.read_file(GEOJSON_PATH)

# Print the shape (number of rows and columns) and CRS of the final GeoDataFrame.
print(f"Shape: {nm_counties.shape}")
print(f"CRS:   {nm_counties.crs}")